# Processamento do Dados

In [2]:
from pyspark.sql import SparkSession, Window
import pyspark.sql.functions as F
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Sessao Spark
spark = SparkSession.builder.appName("ifood-case").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/mnt/HD2/Data_Science/ifood-case/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/07/25 13:31:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
# Leitura do arquivo offers.json
offers = spark.read.option("multiline", "true").json("../data/raw/offers.json")

# Leitura do arquivo profile.json
profile = spark.read.option("multiline", "true").json("../data/raw/profile.json")

# Leitura do arquivo transactions.json
transactions = spark.read.option("multiline", "true").json(
    "../data/raw/transactions.json"
)

In [5]:
# Como vimos, age=118 parece ser um caso onde o cadastro do cliente esta incompleto (EDA 02 - Profile).
# age=118 bate com os nulos de "gender" e "credit_card_limit". Vou converter esses valores para nulos,
# para tratar melhor esses casos e nao distocer algmas estatisticas e modelos.

profile_clean = profile.withColumn(
    "age", F.when(F.col("age") == 118, None).otherwise(F.col("age"))
)

In [6]:
# "registered_on" esta em formato de string (embora a doc diga que deveria ser int).
# Vou converter para "date" para facilitar os calculos de tempo.

profile_clean = profile_clean.withColumn(
    "registered_on", F.to_date(F.col("registered_on"), "yyyyMMdd")
)

In [7]:
# Como visto nas analises, vai ser necessario padronizar as chaves de "offers" e "profile" para poder
# realizar joins.

offers_clean = offers.withColumnRenamed("id", "offer_id")
profile_clean = profile_clean.withColumnRenamed("id", "account_id")

In [8]:
# O campo "value" (struct) de "transactions" possui diversos campos diferentes (exibido na EDA 03).
# Existem duas escritas de offer id: "offer_id" usado para "offer completed" e "offer id" usado para "offer received"
# e "offer viewed". Esses campos vao ser padronizados.

transactions_clean = (
    transactions.withColumn(
        "offer_id", F.coalesce(F.col("value.`offer id`"), F.col("value.offer_id"))
    )
    .withColumn("amount", F.col("value.amount"))
    .withColumn("reward", F.col("value.reward"))
    .drop("value")
)

In [10]:
# Breve validacao
transactions_clean.printSchema()

root
 |-- account_id: string (nullable = true)
 |-- event: string (nullable = true)
 |-- time_since_test_start: double (nullable = true)
 |-- offer_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- reward: double (nullable = true)



In [11]:
# Podemos ver que bate com os resultados da EDA 03
transactions_clean.groupBy("event").agg(
    F.count("amount").alias("amount_nao_nulo"),
    F.count("offer_id").alias("offer_id_nao_nulo"),
    F.count("reward").alias("reward_nao_nulo"),
).show(truncate=False)

+---------------+---------------+-----------------+---------------+
|event          |amount_nao_nulo|offer_id_nao_nulo|reward_nao_nulo|
+---------------+---------------+-----------------+---------------+
|transaction    |138953         |0                |0              |
|offer received |0              |76277            |0              |
|offer completed|0              |33579            |33579          |
|offer viewed   |0              |57725            |0              |
+---------------+---------------+-----------------+---------------+



In [13]:
# Vimos na EDA 03 que exitem ~12k pares de (account_id, offer_id) que possuem mais de uma "offer received"
# Quero manter apenas a primeira ocorrencia de "offer received" para cada par (account_id, offer_id)
# usando "time_since_test_start" como referencia de tempo.

offer_received_window = Window.partitionBy("account_id", "offer_id").orderBy(
    "time_since_test_start"
)

offer_received_dedup = (
    transactions_clean.filter(F.col("event") == "offer received")
    .withColumn("row_number", F.row_number().over(offer_received_window))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

In [14]:
# Confirmando que esta correto, ou seja, que nao existem mais duplicatas de (account_id, offer_id) em "offer received"
offer_received_dedup.groupBy("account_id", "offer_id").count().filter(
    F.col("count") > 1
).count()

0

In [15]:
# Agora, para cada "offer received", vamos buscar os metadados da "offer" e calcular o fim da janela.
# Ou seja, cada "offer received" e o inicio de uma instancia de oferta. Isso vai ser juntado com os metadados de "offer"
# para calcular ate quando foi valida essa oferta. Com isso, vai ser possivel usar essa janela para buscar os eventos de
# "offer viewed" e "offer completed" que aconteceram dentro dessa janela.

offer_instances = (
    offer_received_dedup.withColumnRenamed("time_since_test_start", "received_time")
    .drop("event", "amount", "reward")
    .join(offers_clean, on="offer_id", how="left")
    .withColumn("window_end", F.col("received_time") + F.col("duration"))
)

In [16]:
# Validacao rapida
offer_instances.printSchema()

root
 |-- offer_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- received_time: double (nullable = true)
 |-- channels: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- discount_value: long (nullable = true)
 |-- duration: double (nullable = true)
 |-- min_value: long (nullable = true)
 |-- offer_type: string (nullable = true)
 |-- window_end: double (nullable = true)



In [17]:
offer_instances.show(5, truncate=False)

+--------------------------------+--------------------------------+-------------+----------------------------+--------------+--------+---------+-------------+----------+
|offer_id                        |account_id                      |received_time|channels                    |discount_value|duration|min_value|offer_type   |window_end|
+--------------------------------+--------------------------------+-------------+----------------------------+--------------+--------+---------+-------------+----------+
|f19421c1d4aa40978ebb69ca19b0e20d|0009655768c64bdeb2e877511632db8f|17.0         |[web, email, mobile, social]|5             |5.0     |5        |bogo         |22.0      |
|fafdcd668e3743c1bb461111dcafc2a4|0009655768c64bdeb2e877511632db8f|21.0         |[web, email, mobile, social]|2             |10.0    |10       |discount     |31.0      |
|3f207df678b143eea3cee63160fa8bed|0011e0d4e6b944f998e987f904e8c1e5|0.0          |[web, email, mobile]        |0             |4.0     |0        |inform

In [18]:
# Aqui, para cada linha de "offer_instances", precisamos descobrir se existe um "offer viewed" do mesmo "offer_id"/"account_id"
# dentro da janela de tempo. Caso exista, guardamos o primeiro "viewed_time"

# Extrair os eventos de "offer viewed"
# Aqui, a ideia foi manter so os eventos com "offer viewed" de forma a conter apenas as colunas relevantes
viewed_events = transactions_clean.filter(F.col("event") == "offer viewed").select(
    "account_id", "offer_id", F.col("time_since_test_start").alias("viewed_time")
)

In [29]:
# left join, pois nem toda oferta recebida foi visualizada.
# Cada oferta recebida quero um "offer viewed" do mesmo account_id/offer_id que o viewed_time esteja dentro da janela
oi = offer_instances.alias("oi")
ve = viewed_events.alias("ve")

offer_instances_viewed = (
    oi.join(
        ve,
        on=(
            (F.col("oi.account_id") == F.col("ve.account_id"))
            & (F.col("oi.offer_id") == F.col("ve.offer_id"))
            & (F.col("ve.viewed_time") >= F.col("oi.received_time"))
            & (F.col("ve.viewed_time") <= F.col("oi.window_end"))
        ),
        how="left",
    )
    .drop(F.col("ve.account_id"))
    .drop(F.col("ve.offer_id"))
)

In [30]:
# Como pode existir mais de um "offer viewed" na janela, vai ser mantida apenas a primeira ocorrencia
view_window = Window.partitionBy("account_id", "offer_id").orderBy(
    F.col("viewed_time").asc_nulls_last()
)

In [31]:
offer_instances_viewed = (
    offer_instances_viewed.withColumn("row_number", F.row_number().over(view_window))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

In [32]:
# Verificacao rapida
print(offer_instances_viewed.count(), offer_instances.count())

63288 63288


In [33]:
# Join similar ao passo de cima, mas agora para "offer completed", que existe somente para "bogo"/"discount"
completed_events = transactions_clean.filter(
    F.col("event") == "offer completed"
).select(
    "account_id",
    "offer_id",
    F.col("time_since_test_start").alias("completed_time"),
    "reward",
)

In [34]:
# Padrao igual de join feito acima.
oiv = offer_instances_viewed.alias("oiv")
ce = completed_events.alias("ce")

offer_instances_completed = (
    oiv.join(
        ce,
        on=(
            (F.col("oiv.account_id") == F.col("ce.account_id"))
            & (F.col("oiv.offer_id") == F.col("ce.offer_id"))
            & (F.col("ce.completed_time") >= F.col("oiv.received_time"))
            & (F.col("ce.completed_time") <= F.col("oiv.window_end"))
        ),
        how="left",
    )
    .drop(F.col("ce.account_id"))
    .drop(F.col("ce.offer_id"))
)

In [35]:
# Manter somente a primeira ocorrencia de "offer completed" dentro da janela, caso exista mais de uma
completed_window = Window.partitionBy("account_id", "offer_id").orderBy(
    F.col("completed_time").asc_nulls_last()
)

In [36]:
offer_instances_completed = (
    offer_instances_completed.withColumn(
        "row_number", F.row_number().over(completed_window)
    )
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

In [37]:
# Validacao rapida
print(offer_instances_completed.count(), offer_instances_viewed.count())

63288 63288


In [38]:
# "transaction" nao possui "offer_id", entao o match vai ser por "account_id"
# Com isso, temos que saber se existe algum "transaction" apos o view dentro da janela.
# caso exista, vamos usar isso como uma especie de flag
transaction_events = transactions_clean.filter(F.col("event") == "transaction").select(
    "account_id", F.col("time_since_test_start").alias("transaction_time"), "amount"
)

In [40]:
# Join
oic = offer_instances_completed.alias("oic")
te = transaction_events.alias("te")

# "left_semi" para manter as linhas de "oic" que tem pelo menos um match em "te" (ou seja, que tem pelo menos uma
# transacao dentro da janela)
instances_with_post_view_transaction = oic.join(
    te,
    on=(
        (F.col("oic.account_id") == F.col("te.account_id"))
        & (F.col("te.transaction_time") >= F.col("oic.viewed_time"))
        & (F.col("te.transaction_time") <= F.col("oic.window_end"))
    ),
    how="left_semi",
).select("account_id", "offer_id")

In [41]:
# Adicao da flag booleana para indicar se existe transacao pos-view dentro da janela
offer_instances_completed = offer_instances_completed.join(
    instances_with_post_view_transaction.withColumn(
        "has_post_view_transaction", F.lit(True)
    ),
    on=["account_id", "offer_id"],
    how="left",
).fillna(False, subset=["has_post_view_transaction"])

In [44]:
# Agora vou combinar as informacoes em uma coluna "success"
# bogo/discount: se tiver "offer viewed" E "offer completed" dentro da janela = True, entao sucesso
# informational: se tiver "offer viewed" E "has_post_view_transaction" = True, entao sucesso
offer_instances_labeled = offer_instances_completed.withColumn(
    "success",
    F.when(
        F.col("offer_type").isin("bogo", "discount"),
        F.col("viewed_time").isNotNull()
        & F.col("completed_time").isNotNull()
        & (F.col("completed_time") >= F.col("viewed_time")),
    )
    .when(
        F.col("offer_type") == "informational",
        F.col("viewed_time").isNotNull() & F.col("has_post_view_transaction"),
    )
    .otherwise(False)
    .cast("int"),
)

In [45]:
# Validacao
offer_instances_labeled.groupBy("offer_type").agg(
    F.avg("success").alias("taxa_sucesso"), F.count("*").alias("total")
).show()

+-------------+-------------------+-----+
|   offer_type|       taxa_sucesso|total|
+-------------+-------------------+-----+
|     discount|0.41629799336388057|25316|
|informational|0.38692593470871867|12651|
|         bogo|   0.36337427431776|25321|
+-------------+-------------------+-----+



In [46]:
# Agora, vai ser feito um join com o "profile" para trazer informacoes dos clientes.
# Vai ser um "left" join, pois toda instancia de oferta deve ter um "account_id" correspondente no "profile",
# mas nem todo "account_id" do "profile" tem uma instancia de oferta.
offer_instances_final = offer_instances_labeled.join(
    profile_clean, on="account_id", how="left"
)

In [47]:
# Validacao
print(offer_instances_final.count(), offer_instances_labeled.count())
offer_instances_final.filter(F.col("age").isNull() & F.col("gender").isNull()).count()

63288 63288


8066

In [49]:
# Por fim, selecao das colunas finais que vao ser usadas para analise/modelagem
offer_instances_final = offer_instances_final.select(
    "account_id",
    "offer_id",
    "offer_type",
    "channels",
    "discount_value",
    "min_value",
    "duration",
    "received_time",
    "viewed_time",
    "completed_time",
    "window_end",
    "reward",
    "has_post_view_transaction",
    "success",
    "age",
    "gender",
    "credit_card_limit",
    "registered_on",
)

In [50]:
# Salvar os dados na pasta "processed"
offer_instances_final.write.mode("overwrite").parquet(
    "../data/processed/offer_instances_final.parquet"
)